# Pelatihan Model Prediksi Kalori Terbakar

Notebook ini melatih dan membandingkan dua model untuk memprediksi kalori
terbakar saat olahraga:

1. **Random Forest Regressor** (machine learning klasik)
2. **Neural Network / Multi-Layer Perceptron (MLP)** (deep learning untuk data tabular)

Model dengan performa terbaik kemudian disimpan dan dipakai oleh aplikasi web Flask.

**Dataset:** Kaggle Playground Series S5E5 - "Predict Calorie Expenditure"
https://www.kaggle.com/competitions/playground-series-s5e5/data

## 1. Download Dataset dari Kaggle (tanpa upload file)

**Sebelum menjalankan cell ini, siapkan dulu:**
1. Buka https://www.kaggle.com/settings
2. Scroll ke bagian **API**, klik **Create New Token**
3. File `kaggle.json` akan terdownload -> buka filenya pakai text editor, salin nilai `username` dan `key`
4. Masukkan kedua nilai itu saat diminta di cell berikutnya

In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_USERNAME"] = input("Masukkan Kaggle username: ")
os.environ["KAGGLE_KEY"] = getpass("Masukkan Kaggle API key: ")

!pip install -q kaggle

In [ ]:
!kaggle competitions download -c playground-series-s5e5
!unzip -o playground-series-s5e5.zip

## 2. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

sns.set_style("whitegrid")

data = pd.read_csv("train.csv")
data.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Jumlah baris & kolom:", data.shape)
data.info()

In [ ]:
print("Missing value per kolom:")
print(data.isnull().sum())
print("\nJumlah data duplikat:", data.duplicated().sum())

In [ ]:
data.describe()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=data, x="Sex")
plt.title("Distribusi Jenis Kelamin")
plt.show()

In [ ]:
fitur_numerik = ["Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp", "Calories"]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, kolom in enumerate(fitur_numerik):
    sns.histplot(data[kolom], kde=True, ax=axes[i])
    axes[i].set_title(f"Distribusi {kolom}")
plt.tight_layout()
plt.show()

In [ ]:
data_numerik = data.copy()
data_numerik["Sex"] = data_numerik["Sex"].map({"male": 1, "female": 0})

plt.figure(figsize=(9, 7))
sns.heatmap(data_numerik.drop(columns=["id"]).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap Antar Fitur")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=data.sample(5000, random_state=42), x="Duration", y="Calories", alpha=0.4)
plt.title("Durasi Olahraga vs Kalori Terbakar")
plt.show()

## 4. Preprocessing Data

In [ ]:
from sklearn.model_selection import train_test_split

data["Sex"] = data["Sex"].map({"male": 1, "female": 0})

fitur = ["Sex", "Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]
X = data[fitur]
y = data["Calories"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data latih:", X_train.shape)
print("Data uji:", X_test.shape)

## 5. Model 1 - Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

t0 = time.time()
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)
rf_time = time.time() - t0

rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print(f"Random Forest -> MAE: {rf_mae:.3f} | RMSE: {rf_rmse:.3f} | R2: {rf_r2:.4f} | waktu: {rf_time:.1f}s")

## 6. Model 2 - Neural Network (MLP Regressor)

Neural network butuh fitur discaling dulu (StandardScaler) supaya training stabil,
makanya dibungkus dalam Pipeline supaya scaler ikut tersimpan bersama modelnya.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

t0 = time.time()
nn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        max_iter=100,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=42,
    )),
])
nn_model.fit(X_train, y_train)
nn_time = time.time() - t0

nn_pred = nn_model.predict(X_test)
nn_mae = mean_absolute_error(y_test, nn_pred)
nn_rmse = np.sqrt(mean_squared_error(y_test, nn_pred))
nn_r2 = r2_score(y_test, nn_pred)

print(f"Neural Network -> MAE: {nn_mae:.3f} | RMSE: {nn_rmse:.3f} | R2: {nn_r2:.4f} | waktu: {nn_time:.1f}s")

## 7. Perbandingan Model

In [ ]:
perbandingan = pd.DataFrame({
    "Model": ["Random Forest", "Neural Network (MLP)"],
    "MAE": [rf_mae, nn_mae],
    "RMSE": [rf_rmse, nn_rmse],
    "R2": [rf_r2, nn_r2],
    "Waktu Training (s)": [rf_time, nn_time],
})
perbandingan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(y_test, rf_pred, alpha=0.3, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
axes[0].set_title("Random Forest: Prediksi vs Aktual")
axes[0].set_xlabel("Aktual")
axes[0].set_ylabel("Prediksi")

axes[1].scatter(y_test, nn_pred, alpha=0.3, s=8, color="orange")
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
axes[1].set_title("Neural Network: Prediksi vs Aktual")
axes[1].set_xlabel("Aktual")
axes[1].set_ylabel("Prediksi")

plt.tight_layout()
plt.show()

**Kesimpulan:** Model dengan MAE terkecil dipilih sebagai model final yang
dipakai aplikasi web. Berdasarkan hasil di atas, Neural Network (MLP) memberikan
hasil yang lebih baik dibanding Random Forest pada dataset ini.

## 8. Simpan Model Terbaik

In [ ]:
import joblib

model_terbaik_nama = "Random Forest" if rf_mae < nn_mae else "Neural Network (MLP)"
model_terbaik = rf_model if rf_mae < nn_mae else nn_model
print("Model terbaik:", model_terbaik_nama)

joblib.dump(model_terbaik, "calorie_model.pkl")
print("Model tersimpan sebagai calorie_model.pkl")

from google.colab import files
files.download("calorie_model.pkl")